# 06 · Dashboard de resultados

**Proyecto:** LINE — Auditor Médico Digital · Health & Life IPS SAS

Genera un dashboard HTML **autocontenido** (imágenes embebidas en base64, sin
CDN — abre offline con doble clic) que resume todo el pipeline: limpieza,
validación CUPS, hallazgos EDA y comparación de modelos.

| Entrada | Salida |
|---|---|
| `outputs/reports/*.json`, `eda_hallazgos.md` | `dashboard/dashboard_auditoria.html` |
| `outputs/tables/validacion_cups.csv` | |
| `outputs/figures/*.png` | |

**Regla:** este notebook no inventa números — todo sale de los artefactos
generados por los notebooks 01-05.

## 0 · Configuración (celda autocontenida)

In [1]:
import base64
import json
import os
import time
from pathlib import Path

import pandas as pd

try:
    ROOT = Path(__file__).resolve().parents[1]
except NameError:
    ROOT = Path.cwd()
    if ROOT.name in ("notebooks", "src"):
        ROOT = ROOT.parent

DATA_PROC = ROOT / "data" / "processed"
OUT_REP = ROOT / "outputs" / "reports"
OUT_TAB = ROOT / "outputs" / "tables"
OUT_FIG = ROOT / "outputs" / "figures"
DASH = ROOT / "dashboard"
DASH.mkdir(parents=True, exist_ok=True)

try:
    display  # noqa: B018
except NameError:
    display = print


def reintentar(fn, intentos=6, espera=0.5):
    for _i in range(intentos):
        try:
            return fn()
        except OSError:
            if _i == intentos - 1:
                raise
            time.sleep(espera * (2 ** _i))


def write_text_seguro(path, texto, encoding="utf-8"):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    reintentar(lambda: tmp.write_text(texto, encoding=encoding))
    reintentar(lambda: os.replace(tmp, path))


ENTRADAS = [OUT_REP / "limpieza_reporte.json", OUT_REP / "eda_stats.json",
            OUT_REP / "eda_hallazgos.md", OUT_REP / "metrics.json",
            OUT_TAB / "validacion_cups.csv"]
_faltan = [str(p_) for p_ in ENTRADAS if not p_.exists()]
assert not _faltan, (
    "FALTAN INSUMOS:\n  - " + "\n  - ".join(_faltan)
    + "\n→ Ejecuta primero los notebooks 01 a 05 en orden."
)

limpieza = json.loads((OUT_REP / "limpieza_reporte.json").read_text(encoding="utf-8"))
eda = json.loads((OUT_REP / "eda_stats.json").read_text(encoding="utf-8"))
metrics = json.loads((OUT_REP / "metrics.json").read_text(encoding="utf-8"))
cups = pd.read_csv(OUT_TAB / "validacion_cups.csv")
hallazgos = (OUT_REP / "eda_hallazgos.md").read_text(encoding="utf-8")
print("Insumos OK: reportes, tablas y figuras de los notebooks 01-05")

Insumos OK: reportes, tablas y figuras de los notebooks 01-05


## 1 · Bloques del dashboard
Funciones que convierten los artefactos en HTML: imágenes → base64,
DataFrames y JSON → tablas.

In [2]:
def img64(path):
    path = Path(path)
    if not path.exists():
        return f"<p><em>Figura no encontrada: {path.name}</em></p>"
    b = base64.b64encode(path.read_bytes()).decode()
    return f'<img src="data:image/png;base64,{b}" alt="{path.name}">'


def tabla_metricas():
    filas = []
    for esc, label_esc in [("A_produccion", "A: features de produccion"),
                           ("B_features_cnn", "B: features del CNN")]:
        for nombre, label in [("random_forest", "Random Forest"), ("xgboost", "XGBoost")]:
            r = metrics["escenarios"][esc][nombre]
            for th_key, th_label in [("th_0.50", "0.50"), ("th_recall85", "recall>=85%")]:
                t = r["test"][th_key]
                filas.append(
                    f"<tr><td>{label}</td><td>{label_esc}</td><td>{th_label} "
                    f"(th={t['threshold']})</td><td>{r['test_auc_roc']:.3f}</td>"
                    f"<td>{t['accuracy']:.3f}</td><td>{t['precision']:.3f}</td>"
                    f"<td class='hl'>{t['recall']:.3f}</td><td>{t['f1']:.3f}</td>"
                    f"<td>{r['cv_5fold_train']['roc_auc']}</td></tr>")
    cnn = metrics["cnn_referencia_reportada"]
    filas.append(
        f"<tr class='cnn'><td>CNN MobileNetV2 <b>(corrida de referencia del equipo)</b></td>"
        f"<td>features del CNN</td><td>0.388</td><td>{cnn['test_auc_roc']:.3f}</td>"
        f"<td>{cnn['th_0.388']['accuracy']:.3f}</td><td>{cnn['th_0.388']['precision']:.3f}</td>"
        f"<td class='hl'>{cnn['th_0.388']['recall']:.3f}</td><td>{cnn['th_0.388']['f1']:.3f}</td>"
        f"<td>n/a</td></tr>")
    return "\n".join(filas)


def tabla_cups():
    return "\n".join(
        f"<tr><td><code>{r.codigo_cups}</code></td><td>{r.descripcion}</td>"
        f"<td>{'SI' if r.en_historia_clinica else '-'}</td>"
        f"<td>{'SI' if r.en_prefactura else '-'}</td>"
        f"<td>{r.usos_hc}</td><td>{r.usos_prefactura}</td>"
        f"<td class='{'ok' if r.estado == 'OK' else 'warn'}'>{r.estado}</td></tr>"
        for r in cups.itertuples())


def fk_rotas(info):
    return sum(v for k, v in info.items() if k.startswith("fk_") and isinstance(v, int))


def tabla_limpieza():
    filas = []
    for tabla, info in limpieza.items():
        fk = fk_rotas(info)
        fk_html = f"<span class='warn'>{fk}</span>" if fk else "0"
        filas.append(
            f"<tr><td>{tabla}</td><td>{info['filas']:,}</td>"
            f"<td>{info['duplicados_fila_completa']}</td>"
            f"<td>{info.get('pk_duplicada', 'n/a')}</td>"
            f"<td>{len(info.get('nulos_por_columna', {}))}</td>"
            f"<td>{fk_html}</td></tr>")
    return "\n".join(filas)


total_fk_rotas = sum(fk_rotas(info) for info in limpieza.values())


import re

hallazgos_html = "".join(
    f"<li>{m_.group(1).replace('**', '<b>', 1).replace('**', '</b>', 1)}</li>"
    for l in hallazgos.splitlines()
    if (m_ := re.match(r"^\s*\d+\.\s+(.*)$", l)))

## 2 · Ensamblar y exportar el dashboard → `dashboard/`

In [3]:
html = f"""<!DOCTYPE html>
<html lang="es"><head><meta charset="utf-8">
<title>LINE - Auditor Medico Digital | Dashboard</title>
<style>
 body {{ font-family: 'Segoe UI', system-ui, sans-serif; margin: 0; background: #f4f6f8; color: #263238; }}
 header {{ background: linear-gradient(120deg, #0d47a1, #00838f); color: white; padding: 28px 40px; }}
 header h1 {{ margin: 0 0 6px; font-size: 26px; }}
 main {{ max-width: 1100px; margin: 24px auto; padding: 0 20px; }}
 section {{ background: white; border-radius: 10px; padding: 22px 28px; margin-bottom: 22px;
           box-shadow: 0 1px 4px rgba(0,0,0,.08); }}
 h2 {{ color: #0d47a1; border-bottom: 2px solid #e3f2fd; padding-bottom: 6px; }}
 table {{ border-collapse: collapse; width: 100%; font-size: 13.5px; }}
 th {{ background: #0d47a1; color: white; padding: 7px 9px; text-align: left; }}
 td {{ padding: 6px 9px; border-bottom: 1px solid #eceff1; }}
 tr:hover td {{ background: #f1f8ff; }}
 .hl {{ font-weight: 700; color: #c62828; }}
 .ok {{ color: #2e7d32; font-weight: 600; }}
 .warn {{ color: #ef6c00; font-weight: 700; }}
 .cnn td {{ background: #fff8e1; }}
 img {{ max-width: 100%; border: 1px solid #eceff1; border-radius: 6px; margin: 8px 0; }}
 code {{ background: #eceff1; padding: 1px 5px; border-radius: 4px; }}
 .kpi {{ display: inline-block; background: #e3f2fd; border-radius: 8px; padding: 12px 20px;
        margin: 6px 10px 6px 0; }}
 .kpi b {{ font-size: 22px; color: #0d47a1; display: block; }}
 .nota {{ background: #fff8e1; border-left: 4px solid #f9a825; padding: 10px 14px; border-radius: 4px; }}
</style></head><body>
<header>
 <h1>LINE - Auditor Medico Digital</h1>
 <div>Health &amp; Life IPS SAS - Capstone Samsung Innovation Campus 2025 -
      Generado por el notebook <code>06_dashboard_resultados.ipynb</code></div>
</header>
<main>
<section>
 <h2>1. Resumen</h2>
 <div class="kpi"><b>{eda['target'].get('CONSISTENTE', 0):,}</b>consistentes ({eda['target'].get('CONSISTENTE', 0)/max(sum(eda['target'].values()), 1)*100:.1f}%)</div>
 <div class="kpi"><b>{eda['target'].get('INCONSISTENTE', 0):,}</b>inconsistentes ({eda['target'].get('INCONSISTENTE', 0)/max(sum(eda['target'].values()), 1)*100:.1f}%)</div>
 <div class="kpi"><b>${eda['perdida_estimada_fugas']:,.0f}</b>perdida estimada por {eda['n_fugas']} fugas</div>
 <p>Pipeline: limpieza (nb 01) &rarr; dataset maestro (nb 02) &rarr; validacion CUPS (nb 03)
    &rarr; EDA (nb 04) &rarr; modelos RF/XGBoost (nb 05) &rarr; este dashboard (nb 06).
    Complementos: XGBoost avanzado con SHAP y umbral operativo (nb 07) y CNN transfer
    learning (nb 08).</p>
</section>
<section>
 <h2>2. Limpieza de datos</h2>
 <table>
  <tr><th>Tabla</th><th>Filas</th><th>Duplicados completos</th><th>PK duplicada</th><th>Cols con nulos</th><th>FKs rotas</th></tr>
  {tabla_limpieza()}
 </table>
 <p class="nota"><b>Nulos semanticos conservados:</b> <code>id_prefactura</code> nulo (152) =
 NO_FACTURADO (fuga) e <code>id_detalle_hc</code> nulo (70) = SIN_SOPORTE_CLINICO
 (correspondencia 100% con el tipo de alerta; las otras 87 alertas SIN_SOPORTE tienen fila
 de HC con <code>soporte_clinico = NO</code>). No se eliminaron; se convirtieron en flags.</p>
 {'<p class="nota"><b>Integridad referencial:</b> hay ' + str(total_fk_rotas) + ' fila(s) con llave foránea rota — '
  'las 2 de <code>hc_detalle</code> apuntan a la atención <code>ATN-JEF-000001</code>, inexistente en '
  '<code>atenciones</code>. Se conservan como evidencia: un registro clínico sin atención asociada es en sí '
  'una inconsistencia que Health &amp; Life debe explicar (ver notebooks 01-03).</p>' if total_fk_rotas else ''}
</section>
<section>
 <h2>3. Validacion de codigos CUPS</h2>
 <table>
  <tr><th>Codigo</th><th>Descripcion</th><th>HC</th><th>Prefactura</th><th>Usos HC</th><th>Usos PF</th><th>Estado</th></tr>
  {tabla_cups()}
 </table>
 <p class="nota"><b>Hallazgo:</b> <code>890201-M</code> ("Terapia de rehidratacion oral supervisada")
 no cumple el formato CUPS de 6 digitos pero se usa consistentemente (52 usos HC, 58 prefactura):
 es un codigo interno inventado que cualquier EPS glosaria. REQUIERE_REVISION con Health &amp; Life.
 Los 2 codigos <code>SOLO_HC</code> (<code>890205</code>, <code>902201</code>, 1 uso cada uno) son las 2
 filas de la atencion fantasma <code>ATN-JEF-000001</code> — hallazgo de integridad de datos, no de
 facturacion (ver seccion 2).</p>
</section>
<section>
 <h2>4. Hallazgos EDA</h2>
 <ol>{hallazgos_html}</ol>
 {img64(OUT_FIG / "01_target_alertas.png")}
 {img64(OUT_FIG / "10_flags_cruce.png")}
 {img64(OUT_FIG / "02_tasa_eps.png")}
 {img64(OUT_FIG / "08_serie_temporal.png")}
 {img64(OUT_FIG / "07_valores.png")}
</section>
<section>
 <h2>5. Modelos: Random Forest vs XGBoost vs CNN</h2>
 <table>
  <tr><th>Modelo</th><th>Escenario</th><th>Umbral</th><th>AUC-ROC</th><th>Accuracy</th>
      <th>Precision</th><th>Recall (INCONS.)</th><th>F1</th><th>CV AUC (train)</th></tr>
  {tabla_metricas()}
 </table>
 <p class="nota"><b>Lectura honesta:</b> XGBoost supera al CNN de referencia incluso con las MISMAS
 features (escenario B: efecto del algoritmo). Los flags de cruce de produccion (escenario A)
 suman ~3 puntos mas de AUC. El CNN se mantiene porque el Capstone exige transfer learning;
 RF/XGBoost definen el techo realista para datos tabulares. <b>Pendiente de consolidar:</b> la fila
 CNN cita la corrida original del equipo (AUC 0.749); la corrida guardada del notebook 08 de este
 repositorio reporta AUC &asymp; 0.70 tras fine-tuning — antes de la entrega hay que re-ejecutar el
 nb 08 y dejar una sola cifra.</p>
 {img64(OUT_FIG / "roc_A_produccion.png")}
 {img64(OUT_FIG / "roc_B_features_cnn.png")}
 <h3>Importancia de variables (XGBoost, escenario produccion)</h3>
 {img64(OUT_FIG / "importancias_xgb_A_produccion.png")}
</section>
<section>
 <h2>6. Reproducibilidad</h2>
 <pre><code># desde la raiz del repositorio (github.com/jaoliverosm/Capstone-)
# pip install -r requirements.txt
# abrir Jupyter y ejecutar los notebooks en orden: 01 -> 02 -> 03 -> 04 -> 05 -> 06
# (07 y 08 son complementos: XGBoost avanzado y CNN; el 08 requiere TensorFlow)
# datos originales en data/raw/ (nunca se modifican)
# resultados en data/processed/, outputs/ y dashboard/</code></pre>
 <p class="nota"><b>Estado del CNN:</b> los binarios SI estan en este repositorio
 (<code>outputs/models/cnn/</code>: .h5, .keras y artefactos de preprocesamiento) y el notebook 08
 los regenera de punta a punta. Pendiente: consolidar sus metricas con la corrida de referencia
 (ver seccion 5).</p>
</section>
</main></body></html>
"""

salida = DASH / "dashboard_auditoria.html"
write_text_seguro(salida, html)
print(f"Dashboard generado: {salida} ({salida.stat().st_size/1024:.0f} KB)")

Dashboard generado: C:\Users\ivanp\Desktop\FINAL_CAPSTONE\dashboard\dashboard_auditoria.html (461 KB)


## 3 · Vista rápida de la comparación de modelos

In [4]:
filas = []
for esc, modelos in metrics["escenarios"].items():
    for nombre, r in modelos.items():
        t = r["test"]["th_recall85"]
        filas.append({"escenario": esc, "modelo": nombre, "AUC": r["test_auc_roc"],
                      "recall": t["recall"], "precision": t["precision"], "F1": t["f1"]})
display(pd.DataFrame(filas).sort_values("AUC", ascending=False))

,escenario,modelo,AUC,recall,precision,F1
0,A_produccion,random_forest,0.9185,0.8513,0.6014,0.7049
1,A_produccion,xgboost,0.9152,0.8615,0.5833,0.6957
3,B_features_cnn,xgboost,0.8724,0.8564,0.4154,0.5595
2,B_features_cnn,random_forest,0.8500,0.8513,0.3458,0.4919


## ✅ Verificación de cierre
Debe existir `dashboard/dashboard_auditoria.html` (~0.5 MB, con las imágenes
embebidas). Ábrelo con doble clic — funciona sin internet y sin servidor.
El dashboard debe mostrar **6 hallazgos EDA** y la columna **FKs rotas** en la
tabla de limpieza (con la nota de la atención fantasma `ATN-JEF-000001`).
